In [1]:
import pandas as pd
import csv
import numpy as np

pd.set_option("future.no_silent_downcasting", True)

In [2]:
# import dialogue data
dialogue_raw = pd.read_csv(
    "dialogue/movie_lines.tsv", 
    sep="\t", 
    header=None, 
    names=["lineID", "characterID", "movieID", "character", "text"],
    quoting=csv.QUOTE_NONE,
    encoding="ISO-8859-1",
    on_bad_lines="skip"
)

# import movie data
movies_raw = pd.read_csv(
    "dialogue/movie_titles_metadata.tsv", 
    sep="\t", 
    header=None, 
    names=["movieID", "title", "year", "imdb_rating", "imdb_votes", "genres"],
    quoting=csv.QUOTE_NONE,
    encoding="ISO-8859-1",
    on_bad_lines="skip"
)

# import character data
character_raw = pd.read_csv(
    "dialogue/movie_characters_metadata.tsv", 
    sep="\t", 
    header=None, 
    names=["characterID", "character", "movieID", "title", "gender", "position"],
    quoting=csv.QUOTE_NONE,
    encoding="ISO-8859-1",
    on_bad_lines="skip"
)   

# import Oscar nominees data
oscars_raw = pd.read_csv("oscars/the_oscar_award.csv", header=0)

In [3]:
# clean Oscars data
oscars = oscars_raw.dropna(subset=["film"]).copy()
oscars.loc[:, "title"] = oscars_raw["film"].str.replace(r'[^A-Za-z0-9 ]', '', regex=True).str.lower().str.strip()
oscars.loc[:, "year"] = (oscars_raw["year_film"])
oscars.loc[:, "award"]= oscars_raw["canon_category"].astype("category")

# clean dialogue data
dialogue_raw["lineID"] = dialogue_raw["lineID"].str.strip('"')
dialogue_raw["characterID"] = dialogue_raw["characterID"].str.strip('"')
dialogue_raw["movieID"] = dialogue_raw["movieID"].str.strip('"')
dialogue_raw["character"] = dialogue_raw["character"].str.strip('"')
dialogue_raw["text"] = dialogue_raw["text"].str.strip()

# clean movie data
movies = movies_raw[["movieID", "title", "year", "genres"]].copy()
movies["year"] = movies["year"].str.replace(r'[^0-9 ]', '', regex=True).str.lower().str.strip()
movies["year"] = movies["year"].astype("Int64")

# clean character data
characters = character_raw["characterID"].copy()

In [4]:
# group oscars data by title and year and tally awards
oscars = oscars.groupby(["title", "year"]).agg({
    "award": lambda x: len(x.unique())
}).reset_index()

movies as a whole
vs movies with oscar nominations
vs movies without oscar nominations

In [14]:
# join the dataframes
dialogue = dialogue_raw.merge(movies, on="movieID", how="left")
dialogue = dialogue.merge(characters, on="characterID", how="left")
dialogue = dialogue.merge(oscars, on=["title", "year"], how="left")

# drop rows with missing dialogue text
dialogue = dialogue.dropna(subset=["text"])

In [ ]:
import nltk
# nltk.download("punkt_tab")
import string
from nltk.tokenize import word_tokenize

# tokenize dialogue text and remove punctuation
translator = str.maketrans('', '', string.punctuation)
dialogue["text"] = dialogue["text"].apply(lambda x: x.translate(translator))

dialogue["tokens"] = dialogue["text"].str.lower().apply(word_tokenize)

In [35]:
# split dataframes for movies with and without Oscar nominations
nominated = dialogue[dialogue["award"].notna()].copy()
no_noms = dialogue[dialogue["award"].isna()].copy()

In [ ]:
# movies with at least one Oscar nomination
fdist_noms = nltk.FreqDist(nominated["tokens"].explode())

# drop stop words
stop_words = set(nltk.corpus.stopwords.words("english"))
stop_words.update(string.punctuation)
fdist_noms = {word: freq for word, freq in fdist_noms.items() if word not in stop_words}

# filter out shorter words
fdist_noms = {word: freq for word, freq in fdist_noms.items() if len(str(word)) >= 4}

# get most common words in nominated movies
top_noms = dict(sorted(fdist_noms.items(), key=lambda item: item[1], reverse=True)[:100])
# top_noms

In [49]:
# movies with no Oscar nominations
fdist_none = nltk.FreqDist(no_noms["tokens"].explode())

# drop stop words
fdist_none = {word: freq for word, freq in fdist_none.items() if word not in stop_words}

# filter out shorter words
fdist_none = {word: freq for word, freq in fdist_none.items() if len(str(word)) >= 4}

# get most common words in movies with no nominations
top_none = dict(sorted(fdist_none.items(), key=lambda item: item[1], reverse=True)[:100])
# top_none

In [ ]:
## find common words in both sets
# common_words = set(top_noms.keys()) & set(top_none.keys())
# common_words

# find unique words in each set
unique_noms = set(top_noms.keys()) - set(top_none.keys())
unique_none = set(top_none.keys()) - set(top_noms.keys())
unique_noms, unique_none

({'aint',
  'every',
  'fine',
  'girl',
  'hear',
  'leave',
  'thank',
  'three',
  'understand',
  'wouldnt'},
 {'dead',
  'fucking',
  'guys',
  'happened',
  'kill',
  'might',
  'shit',
  'talking',
  'wait',
  'wrong'})

NLP of dialogue responses Oscar noms vs none